In [69]:
# load .rds saved datasets into variables
expr <- readRDS("../datasets/expression_matrix_train.rds")
clinical <- readRDS("../datasets/clinical_metadata_train.rds")

head(expr)
dim(expr)

tail(clinical)
dim(clinical)

Gene Symbol,GSM1045191,GSM1045192,GSM1045193,GSM1045194,GSM1045195,GSM1045196,GSM1045197,GSM1045198,GSM1045199,⋯,GSM1045302,GSM1045303,GSM1045304,GSM1045305,GSM1045306,GSM1045307,GSM1045308,GSM1045309,GSM1045310,GSM1045311
<chr>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,⋯,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>
A1BG,5.061765,5.384287,5.462018,5.104647,4.940101,5.087535,5.647719,5.502169,4.935021,⋯,5.612234,5.074677,6.872525,8.020731,5.132484,5.815884,5.310448,5.293781,5.683640,5.880834
A1BG-AS1,4.213439,4.497735,5.442854,4.142935,3.836375,3.840813,4.108047,4.117811,4.374573,⋯,5.274847,4.386748,5.158049,4.650182,4.528774,4.627988,4.577176,4.533182,4.283014,4.635586
A1CF,2.960794,3.844863,3.603847,3.710878,2.919968,3.246907,2.954523,3.181512,3.115969,⋯,3.695726,3.247118,3.037334,3.657515,3.350718,2.967194,2.943969,3.404410,3.238819,2.898562
A2M,6.814863,6.845371,4.532851,7.000975,7.117619,6.683618,6.839982,7.182724,6.365128,⋯,5.206523,6.804822,6.542975,5.768323,5.881436,6.835486,6.455288,6.328355,6.584059,6.576534
A2M-AS1,4.973031,4.252506,3.920574,4.064233,5.378826,5.553602,5.198546,5.391066,5.675315,⋯,3.974318,4.694706,4.407413,3.791085,3.544270,4.649494,3.700493,3.785728,3.637337,4.267453
A2ML1,3.133812,3.408455,3.779359,3.546180,3.138060,3.030034,3.338255,3.241485,3.111028,⋯,4.034024,3.141456,3.195094,3.162890,3.291446,2.894613,2.764031,3.472512,3.094196,3.243091


[1] 21655   122

,sample_id,tissue_type,patient_age,tumor_grade,tumor_size,er_status,lymph_node_status,relapse_free_event,relapse_free_days
,<chr>,<chr>,<chr>,<chr>,<chr>,<chr>,<chr>,<chr>,<chr>
GSM1045306,GSM1045306,breast cancer,60.39,3,6,0,1,1,634
GSM1045307,GSM1045307,breast cancer,46.09,2,2,1,1,0,2952
GSM1045308,GSM1045308,breast cancer,47.82,1,3,NA,1,1,1209
GSM1045309,GSM1045309,breast cancer,48.43,2,3.5,1,1,0,2105
GSM1045310,GSM1045310,breast cancer,70.44,2,2,1,1,1,639
GSM1045311,GSM1045311,breast cancer,62.51,2,1.5,1,1,0,2962


[1] 121   9

Perform t-test on normal vs tumor for each gene

Evaluate if significant p-value

In [70]:
# separate sample ids into normal and tumor
normal_ids <- clinical$sample_id[clinical$tissue == "normal breast"]
tumor_ids <- clinical$sample_id[clinical$tissue == "breast cancer"]

length(normal_ids)
length(tumor_ids)

[1] 17

[1] 104

Doing first gene manually for learning purposes

In [71]:
# get all values for gene 1: A1BG
gene1_values <- as.numeric(expr[1, -1])
names(gene1_values) <- colnames(expr)[-1]

# split into normal vs tumor
normal_values <- gene1_values[normal_ids]
tumor_values <- gene1_values[tumor_ids]

In [72]:
# performing a standard t-test

# calcultate means for both
normal_mean <- mean(normal_values)
tumor_mean <- mean(tumor_values)

# find variance
normal_variance <- sum((normal_values - normal_mean)^2) / (length(normal_values) - 1)
tumor_variance <- sum((tumor_mean - tumor_values)^2) / (length(tumor_values) - 1)

# find standard error
se <- sqrt((normal_variance / length(normal_values)) + (tumor_variance / length(tumor_values)))

# calculate t-test
t <- (tumor_mean - normal_mean) / (se)

normal_mean
tumor_mean
normal_variance
tumor_variance
se
t

[1] 5.18546

[1] 5.710515

[1] 0.08178687

[1] 0.3966322

[1] 0.0928696

[1] 5.653681

In [73]:
# calculate degrees of freedom
deg_free <- length(normal_values) + length(tumor_values) - 2

# Get p-value (two-tailed test)
p_value <- 2 * pt(-abs(t), deg_free)

p_value

[1] 1.096328e-07

In [74]:
# validate my output with Welch's t test function
test <- t.test(tumor_values, normal_values)

test$p.value

[1] 9.046909e-07

Performing Benjamini-Hochberg method on 10 t-test genes manually (for learning purposes)

In [ ]:
# defining a function to perform a manual Benjamini-Hochberg
manual_BH <- function(pvalues, alpha = 0.05) {
  m <- length(pvalues)
  
  # Step 1: Sort
  sorted_indices <- order(pvalues)
  sorted_p <- pvalues[sorted_indices]
  
  # Step 2: Rank sorted p values
  ranks <- 1:m

  # Step 3: Calculate adjusted p-values
  adjusted <- (m / ranks) * sorted_p
  
  # Step 4: Monotonicity
  for (i in (m - 1):1) {
    adjusted[i] <- min(adjusted[i], adjusted[i + 1])
  }
  adjusted <- pmin(adjusted, 1)
  
  # Step 5: Unsort to original order
  fdr <- numeric(m)
  fdr[sorted_indices] <- adjusted
  
  return(fdr)
}

In [76]:
# perform t-test on 10 genes
pvalues <- numeric(10)

for (i in 1:10) {
    # get all values for gene i
    gene_values <- as.numeric(expr[i, -1])
    names(gene_values) <- colnames(expr)[-1]

    # split into normal vs tumor
    normal_values <- gene_values[normal_ids]
    tumor_values <- gene_values[tumor_ids]

    # perform t-test
    test <- t.test(tumor_values, normal_values)

    # calculate degrees of freedom
    deg_free <- length(normal_values) + length(tumor_values) - 2

    # get p-value (two-tailed test)
    pvalues[[i]] <- 2 * pt(-abs(t), deg_free)
}
pvalues

# manual Benjamini-Hochberg formula
manual_BH(pvalues)

[1] 1.096328e-07 1.096328e-07 1.096328e-07 1.096328e-07 1.096328e-07
 [6] 1.096328e-07 1.096328e-07 1.096328e-07 1.096328e-07 1.096328e-07

[1] 1.096328e-07 1.096328e-07 1.096328e-07 1.096328e-07 1.096328e-07
 [6] 1.096328e-07 1.096328e-07 1.096328e-07 1.096328e-07 1.096328e-07

In [77]:
# validate with built in function
fdr <- p.adjust(pvalues, method = "BH")
fdr

[1] 1.096328e-07 1.096328e-07 1.096328e-07 1.096328e-07 1.096328e-07
 [6] 1.096328e-07 1.096328e-07 1.096328e-07 1.096328e-07 1.096328e-07

Calculate Fold Change manually: look for biologically meaninful ratio of gene expression

In [78]:
# get all values for gene 1: A1BG
gene1_values <- as.numeric(expr[1, -1])
names(gene1_values) <- colnames(expr)[-1]

# split into normal vs tumor
normal_values <- gene1_values[normal_ids]
tumor_values <- gene1_values[tumor_ids]

# calcultate means for both
normal_mean <- mean(normal_values)
tumor_mean <- mean(tumor_values)

# find log base 2 for fold change
logFC <- log2(tumor_mean / normal_mean)
logFC

# gene 1 is biologically insignificant

[1] 0.1391489